In [4]:
#!pip install statsmodels


   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------- ----------- 6.8/9.5 MB 35.0 MB/s eta 0:00:01
   ---------------------------------------  9.4/9.5 MB 34.6 MB/s eta 0:00:01
   ---------------------------------------  9.4/9.5 MB 34.6 MB/s eta 0:00:01
   ---------------------------------------  9.4/9.5 MB 34.6 MB/s eta 0:00:01
   ---------------------------------------  9.4/9.5 MB 34.6 MB/s eta 0:00:01
   ---------------------------------------- 9.5/9.5 MB 9.0 MB/s eta 0:00:00


In [1]:
"""
AR Model Diagnostic — Full Sample Specification Analysis
=========================================================
Lag order analysis to inform AR specification choices.
Produces: summary stats, AIC/BIC by lag, ACF/PACF, recursive AIC at 3 windows.
Not forecasting — no future data used.
Output: Output_excel.xlsx, Output_plots.png
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.stats.stattools import durbin_watson
import warnings, os
warnings.filterwarnings("ignore")

INPUT_FILE  = "Input_TTF_NG_Real_Average_Prices.xlsx"
OUT_EXCEL   = "Output_excel.xlsx"
OUT_PLOT    = "Output_plots.png"
P_MAX       = 24


In [2]:
df = pd.read_excel(INPUT_FILE, parse_dates=["date"])
df["log_price"] = np.log(df["price_real"])
y = df["log_price"].values
dates = df["date"].values
T = len(y)

print(f"Sample: {df['date'].iloc[0].strftime('%d/%m/%Y')} — {df['date'].iloc[-1].strftime('%d/%m/%Y')}")
print(f"Observations: {T}")


Sample: 28/02/2006 — 31/12/2025
Observations: 239


In [3]:
log_diff = np.diff(y)

stats = {
    "Statistic": [
        "Observations (levels)", "Mean (log price)", "Std Dev (log price)",
        "Min (log price)", "Max (log price)",
        "Mean (log return)", "Std Dev (log return)",
        "First date", "Last date",
    ],
    "Value": [
        T,
        round(y.mean(), 6), round(y.std(), 6),
        round(y.min(), 6),  round(y.max(), 6),
        round(log_diff.mean(), 6), round(log_diff.std(), 6),
        df["date"].iloc[0].strftime("%d/%m/%Y"),
        df["date"].iloc[-1].strftime("%d/%m/%Y"),
    ]
}
stats_df = pd.DataFrame(stats)


In [4]:
aic_rows = []
for p in range(1, P_MAX + 1):
    try:
        model = AutoReg(y, lags=p, old_names=False).fit()
        aic_rows.append({
            "Lag order p": p,
            "AIC":         round(model.aic, 4),
            "BIC":         round(model.bic, 4),
            "Log-Lik":     round(model.llf, 4),
            "Num params":  p + 1,
            "Obs used":    T - p,
        })
    except Exception:
        aic_rows.append({"Lag order p": p, "AIC": np.nan, "BIC": np.nan,
                         "Log-Lik": np.nan, "Num params": p+1, "Obs used": T-p})

aic_df = pd.DataFrame(aic_rows)
aic_selected = int(aic_df.loc[aic_df["AIC"].idxmin(), "Lag order p"])
bic_selected = int(aic_df.loc[aic_df["BIC"].idxmin(), "Lag order p"])

print(f"Full-sample AIC selects: AR({aic_selected})")
print(f"Full-sample BIC selects: AR({bic_selected})")
print(aic_df.sort_values("AIC").head(10).to_string(index=False))


Full-sample AIC selects: AR(20)
Full-sample BIC selects: AR(1)
 Lag order p       AIC       BIC  Log-Lik  Num params  Obs used
          20 -210.5441 -135.9846 127.2721          21       219
          18 -207.4529 -139.4896 123.7264          19       221
          21 -206.6015 -128.7581 126.3008          22       218
          19 -205.2899 -134.0237 123.6450          20       220
          22 -204.3546 -123.2371 126.1773          23       217
          24 -201.7618 -114.1252 126.8809          25       215
          23 -200.2236 -115.8416 125.1118          24       216
          16 -195.3355 -134.0064 115.6677          17       223
          13 -194.2664 -142.9584 112.1332          14       226
          15 -192.9702 -134.9722 113.4851          16       224


In [5]:
n_lags = 24
acf_vals,  acf_ci  = acf(y,  nlags=n_lags, alpha=0.05)
pacf_vals, pacf_ci = pacf(y, nlags=n_lags, alpha=0.05, method="ols")

acf_df = pd.DataFrame({
    "Lag":         range(0, n_lags + 1),
    "ACF":         [round(v, 6) for v in acf_vals],
    "ACF CI low":  [round(v[0] - acf_vals[i], 6) for i, v in enumerate(acf_ci)],
    "ACF CI high": [round(v[1] - acf_vals[i], 6) for i, v in enumerate(acf_ci)],
    "PACF":        [round(v, 6) for v in pacf_vals],
    "PACF CI low": [round(v[0] - pacf_vals[i], 6) for i, v in enumerate(pacf_ci)],
    "PACF CI high":[round(v[1] - pacf_vals[i], 6) for i, v in enumerate(pacf_ci)],
})


In [6]:
# Recursive AIC at 3 windows: early (2015), mid (2019), late (2023)
target_dates = ["2015-01-31", "2019-06-30", "2023-12-31"]
recursive_rows = []

for td in target_dates:
    sub = df[df["date"] <= pd.Timestamp(td)]["log_price"].values
    T_sub = len(sub)
    best_aic, best_p     = np.inf, None
    best_bic, best_p_bic = np.inf, None
    for p in range(1, min(P_MAX, T_sub // 4) + 1):
        try:
            m = AutoReg(sub, lags=p, old_names=False).fit()
            if m.aic < best_aic: best_aic, best_p     = m.aic, p
            if m.bic < best_bic: best_bic, best_p_bic = m.bic, p
        except:
            pass
    recursive_rows.append({
        "Window end":    td,
        "Obs available": T_sub,
        "AIC selects p": best_p,
        "AIC value":     round(best_aic, 4),
        "BIC selects p": best_p_bic,
        "BIC value":     round(best_bic, 4),
    })
    print(f"Window to {td}: T={T_sub}, AIC→AR({best_p}), BIC→AR({best_p_bic})")

recursive_df = pd.DataFrame(recursive_rows)


Window to 2015-01-31: T=108, AIC→AR(18), BIC→AR(1)
Window to 2019-06-30: T=161, AIC→AR(18), BIC→AR(1)
Window to 2023-12-31: T=215, AIC→AR(18), BIC→AR(1)


In [9]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("AR Model Diagnostic - Log Real TTF NG Price"
             f"Full sample: Feb 2006 – {df['date'].iloc[-1].strftime('%b %Y')}",
             fontsize=13, fontweight="bold")

ax1 = axes[0, 0]
ax1.plot(df["date"], y, color="#1F4E79", linewidth=1.2)
ax1.axvline(pd.Timestamp("2015-01-31"), color="green",
            linestyle="--", linewidth=1, label="Eval start (Jan 2015)")
ax1.set_title("Log Real TTF Price", fontsize=11)
ax1.set_ylabel("ln(Real Price)")
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3)

ax2 = axes[0, 1]
colors = ["#C00000" if p == aic_selected else "#2E75B6" for p in aic_df["Lag order p"]]
ax2.bar(aic_df["Lag order p"], aic_df["AIC"], color=colors, edgecolor="white")
ax2.axvline(aic_selected, color="#C00000", linestyle="--", linewidth=1.5,
            label=f"AIC min → AR({aic_selected})")
ax2.axvline(bic_selected, color="orange", linestyle="--", linewidth=1.5,
            label=f"BIC min → AR({bic_selected})")
ax2.set_title("AIC and BIC by Lag Order (Full Sample)", fontsize=11)
ax2.set_xlabel("Lag order p")
ax2.set_ylabel("Information Criterion")
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3, axis="y")

conf = 1.96 / np.sqrt(T)
ax3 = axes[1, 0]
ax3.bar(acf_df["Lag"][1:], acf_df["ACF"][1:],
        color=["#C00000" if abs(v) > conf else "#2E75B6" for v in acf_df["ACF"][1:]])
ax3.axhline(conf,  color="red", linestyle="--", linewidth=1, label="95% CI")
ax3.axhline(-conf, color="red", linestyle="--", linewidth=1)
ax3.axhline(0,     color="black", linewidth=0.5)
ax3.set_title("ACF — Log Real Price", fontsize=11)
ax3.set_xlabel("Lag")
ax3.set_ylabel("ACF")
ax3.legend(fontsize=8)
ax3.grid(alpha=0.3, axis="y")


In [10]:
ax4 = axes[1, 1]
ax4.bar(acf_df["Lag"][1:], acf_df["PACF"][1:],
        color=["#C00000" if abs(v) > conf else "#2E75B6" for v in acf_df["PACF"][1:]])
ax4.axhline(conf,  color="red", linestyle="--", linewidth=1, label="95% CI")
ax4.axhline(-conf, color="red", linestyle="--", linewidth=1)
ax4.axhline(0,     color="black", linewidth=0.5)
ax4.set_title("PACF — Log Real Price", fontsize=11)
ax4.set_xlabel("Lag")
ax4.set_ylabel("PACF")
ax4.legend(fontsize=8)
ax4.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(OUT_PLOT, dpi=150, bbox_inches="tight")
plt.close()
print(f"Plots saved to: {OUT_PLOT}")


Plots saved to: Output_plots.png


In [11]:
with pd.ExcelWriter(OUT_EXCEL, engine="openpyxl") as writer:
    stats_df.to_excel(writer,     sheet_name="Summary Stats",          index=False)
    aic_df.to_excel(writer,       sheet_name="AIC by Lag Order",       index=False)
    acf_df.to_excel(writer,       sheet_name="ACF and PACF",           index=False)
    recursive_df.to_excel(writer, sheet_name="Recursive AIC Windows",  index=False)

print(f"Saved to: {OUT_EXCEL}")
print(f"Full-sample AIC → AR({aic_selected}),  BIC → AR({bic_selected})")
for row in recursive_rows:
    print(f"  To {row['Window end']}: AIC→AR({row['AIC selects p']}), BIC→AR({row['BIC selects p']})")


Saved to: Output_excel.xlsx
Full-sample AIC → AR(20),  BIC → AR(1)
  To 2015-01-31: AIC→AR(18), BIC→AR(1)
  To 2019-06-30: AIC→AR(18), BIC→AR(1)
  To 2023-12-31: AIC→AR(18), BIC→AR(1)
